# Generate 1 GB test CSVs in the Lakehouse
Python notebook. Writes one ~1 GB CSV per feed straight into `Files/landing/<feed>/` of the **default Lakehouse**, so nothing has to be uploaded. Writes in 2-million-row chunks, so memory stays low; all three files took under a minute in testing outside Fabric.

Same columns, formats and seed as the downloadable `*_20261017.csv` files; ids continue on from the 100 MB set.

In [ ]:
LANDING   = "/lakehouse/default/Files/landing"  # default-Lakehouse mount
TARGET_GB = 1.0                                   # size per file
FILE_DATE = "20261017"                            # goes in each file name

In [ ]:
import numpy as np, polars as pl, os
rng = np.random.default_rng(11)
TARGET = int(TARGET_GB * 1024**3); CHUNK = 2_000_000
first=np.array(["Alice","Ben","Chloe","Dan","Ella","Finn","Grace","Harry","Isla","Jack","Kate","Liam","Mia","Noah","Olivia","Sam"])
last=np.array(["Smith","Jones","Taylor","Brown","Wilson","Evans","Thomas","Roberts","Walker","Wright"])
countries=np.array(["United Kingdom","Ireland","France","Germany","Netherlands","Spain","United States"])
adj=np.array(["Classic","Premium","Eco","Compact","Deluxe","Smart","Pro","Mini"])
noun=np.array(["Kettle","Lamp","Chair","Desk","Backpack","Headphones","Mug","Speaker","Monitor","Keyboard"])
def customers(start,n,_):
    ts=(np.datetime64("2024-01-01T00:00:00")+rng.integers(0,86400*650,n).astype("timedelta64[s]")).astype("datetime64[ms]")
    return pl.DataFrame({"customer_id":np.arange(start,start+n),
        "customer_name":np.char.add(np.char.add(rng.choice(first,n)," "),rng.choice(last,n)),
        "country":rng.choice(countries,n),"created_at":ts}).with_columns(pl.col("created_at").dt.strftime("%Y-%m-%d %H:%M:%S"))
def orders(start,n,maxcust):
    d=np.datetime64("2026-01-01")+rng.integers(0,288,n).astype("timedelta64[D]")
    return pl.DataFrame({"order_id":np.arange(start,start+n),"customer_id":rng.integers(1,maxcust+1,n),
        "order_date":d,"amount":np.round(rng.uniform(5,1200,n),2)}).with_columns(
        pl.col("order_date").dt.strftime("%Y-%m-%d"),pl.col("amount").cast(pl.Decimal(18,2)).cast(pl.Utf8))
def products(start,n,_):
    return pl.DataFrame({"product_id":np.arange(start,start+n),
        "product_name":np.char.add(np.char.add(rng.choice(adj,n)," "),rng.choice(noun,n)),
        "unit_price":np.round(rng.uniform(2.5,450,n),2)}).with_columns(pl.col("unit_price").cast(pl.Decimal(18,2)).cast(pl.Utf8))
def build(make,folder,name,start,extra=None):
    p=os.path.join(LANDING, folder, name); os.makedirs(os.path.dirname(p), exist_ok=True); nid=start; header=True
    with open(p,"wb") as f:
        while True:
            size=f.tell(); remaining=TARGET-size
            if remaining<=0: break
            df=make(nid,CHUNK,extra)
            buf=df.write_csv(include_header=header).encode()
            per=len(buf)/CHUNK
            if len(buf)>remaining:
                n=max(1,int(remaining/per)); df=df.head(n); buf=df.write_csv(include_header=header).encode()
            f.write(buf); nid+=df.height; header=False
            if df.height<CHUNK: break
    print(name, nid-start, "rows", round(os.path.getsize(p)/1024**2), "MiB", "ids", start, "-", nid-1); return nid-1
lastc=build(customers,"customers",f"customers_{FILE_DATE}.csv",2197671)
build(orders,"orders",f"orders_{FILE_DATE}.csv",3323329,lastc)
build(products,"products",f"products_{FILE_DATE}.csv",4017128)
